# 1. Carga y Limpieza Base Socioeconómica
Cargamos los datos de los barrios y eliminamos los identificadores basura y columnas masivamente nulas (nacionalidades).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

df_socio = pd.read_csv('../data/processed/barrios_features.csv')
print(f'Original: {df_socio.shape}')

# Eliminar columnas con más del 50% de nulos (banderas/nacionalidades extremas)
missing = df_socio.isnull().mean()
cols_a_borrar = missing[missing > 0.5].index.tolist()
df_socio.drop(columns=cols_a_borrar, inplace=True)
print(f'Tras purga de nulos: {df_socio.shape}')
df_socio.fillna(0, inplace=True)


# 2. Ingeniería de Atributos (Super-Métricas o Scores)
Convertimos decenas de métricas crudas en 4 perfiles principales puntuados de 0 a 100 para que el algoritmo Recommender de Streamlit los digiera fácilmente.

In [ ]:
# Inicializar Scaler de 0 a 100
scaler = MinMaxScaler(feature_range=(0, 100))

# 1. Vitalidad Joven/Moderna (15 a 44 años)
df_socio['score_joven'] = df_socio['Población de 15 a 29 años'] + df_socio['Población de 30 a 44 años']

# 2. Vitalidad Senior (Mayores de 65)
if 'Población de 65 años y más' in df_socio.columns:
    df_socio['score_senior'] = df_socio['Población de 65 años y más']
else:
    df_socio['score_senior'] = df_socio['Población de 65 a 79 años'] + df_socio.get('Población de 80 años y más', 0)

# 3. Poder Adquisitivo (Proxy: Alta educación penalizando el Paro)
col_univ = [c for c in df_socio.columns if 'estudios superiores' in c.lower()][0]
df_socio['score_adquisitivo'] = df_socio[col_univ] - df_socio['Tasa absoluta de paro registrado (febrero)']

# 4. Tráfico Familiar (Menores de 14 años)
df_socio['score_familiar'] = df_socio['Población de 0 a 14 años']

# Escalar todo al rango 0-100 para que sean puntuables limpios
scores_cols = ['score_joven', 'score_senior', 'score_adquisitivo', 'score_familiar']
df_socio[scores_cols] = scaler.fit_transform(df_socio[scores_cols])
df_socio[scores_cols] = df_socio[scores_cols].round(2)

# Aislar el DataFrame limpio final con solo la pulpa
cols_finales = ['distrito', 'barrio', 'Población densidad (hab./ha.)'] + scores_cols
df_socio_clean = df_socio[cols_finales].copy()
df_socio_clean.head(5)


# 3. Mezcla Final con Locales
Sustituimos el ruidero socioeconómico crudo en el oficial de locales por estos super scores, listos para tu `recommender.py` y `app.py`.

In [ ]:
import os

maestro_path = '../data/processed/negocios_maestro.csv'
if os.path.exists(maestro_path):
    df_locales = pd.read_csv(maestro_path)
    
    # Limpieza de strings para matching exacto
    df_locales['desc_barrio_local'] = df_locales['desc_barrio_local'].astype(str).str.strip().str.upper()
    df_socio_clean['barrio_match'] = df_socio_clean['barrio'].astype(str).str.strip().str.upper()
    
    # Limpiar columnas viejas socioeconómicas si ya estaban en el CSV anterior
    cols_viejas = [c for c in df_socio.columns if c in df_locales.columns and c not in ['barrio', 'distrito', 'desc_barrio_local']]
    df_locales = df_locales.drop(columns=cols_viejas, errors='ignore')

    # Merge Definitivo
    df_scored = pd.merge(df_locales, df_socio_clean, left_on='desc_barrio_local', right_on='barrio_match', how='left')
    df_scored.drop(columns=['barrio_match'], inplace=True)
    
    # Guardado del Set Final Oro puro
    output_file = '../data/processed/negocios_scored.csv'
    df_scored.to_csv(output_file, index=False)
    
    print(f'✅ DATASET FINAL GUARDADO EN {output_file}')
    print(f'Total Registros: {len(df_scored)} locales activos.')
    print(f'Variables (columnas finalizadas): {len(df_scored.columns)}')
else:
    print('Aviso: Ejecuta el notebook 01_exploracion_y_merged primero para generar negocios_maestro.csv')
